# Qdrant vector database basics

This notebook introduces [Qdrant](https://qdrant.tech/), a vector database for storing vectors with payloads and searching by vector similarity.

The vectors below are small, hand-written teaching data. In a real retrieval application, create document and query vectors with the same embedding model. The notebook uses Qdrant's in-memory mode, so it does not require Docker or a running server.

Install the dependency from the repository root if needed:

```bash
./.venv/bin/pip install qdrant-client
```

The example uses Qdrant's in-memory mode, so it does not require Docker or a
running server. It creates a collection, inserts points, runs similarity and
filtered searches, updates a payload, retrieves a point, and deletes a point.

To connect to a Qdrant server instead, start one locally:

```bash
docker run --rm -p 6333:6333 qdrant/qdrant
./.venv/bin/python quadrant/ex01_qdrant_basics.py --url http://localhost:6333
```

## What the example teaches

- A Qdrant point has an ID, a vector, and an optional payload.
- A collection's vector size and distance metric are configured up front.
- `query_points` returns the nearest points for a query vector.
- Payload filters narrow a vector search to matching records.
- The four-dimensional vectors are hand-written teaching data, not a real
  embedding model. In a retrieval application, create document and query
  vectors with the same embedding model before sending them to Qdrant.


## 1. Imports and collection settings

In [1]:
from pydantic import BaseModel
from qdrant_client import QdrantClient
from qdrant_client.http import models

COLLECTION_NAME = "learning_resources"
VECTOR_SIZE = 4

## 2. Define records and teaching vectors

A Qdrant point contains an ID, a vector, and an optional payload. Pydantic keeps the source record explicit before it is transformed into Qdrant points.

In [2]:
class LearningResource(BaseModel):
    point_id: int
    title: str
    text: str
    topic: str
    vector: list[float]


resources = [
    LearningResource(
        point_id=1,
        title="Python lists",
        text="Lists store an ordered collection of Python values.",
        topic="python",
        vector=[0.95, 0.05, 0.00, 0.00],
    ),
    LearningResource(
        point_id=2,
        title="Python dictionaries",
        text="Dictionaries map keys to values for fast lookup.",
        topic="python",
        vector=[0.90, 0.10, 0.00, 0.00],
    ),
    LearningResource(
        point_id=3,
        title="Vector databases",
        text="Vector databases find records by embedding similarity.",
        topic="databases",
        vector=[0.05, 0.95, 0.00, 0.00],
    ),
    LearningResource(
        point_id=4,
        title="SQL indexes",
        text="SQL indexes speed up lookups on structured columns.",
        topic="databases",
        vector=[0.00, 0.85, 0.15, 0.00],
    ),
]
resources

[LearningResource(point_id=1, title='Python lists', text='Lists store an ordered collection of Python values.', topic='python', vector=[0.95, 0.05, 0.0, 0.0]),
 LearningResource(point_id=2, title='Python dictionaries', text='Dictionaries map keys to values for fast lookup.', topic='python', vector=[0.9, 0.1, 0.0, 0.0]),
 LearningResource(point_id=3, title='Vector databases', text='Vector databases find records by embedding similarity.', topic='databases', vector=[0.05, 0.95, 0.0, 0.0]),
 LearningResource(point_id=4, title='SQL indexes', text='SQL indexes speed up lookups on structured columns.', topic='databases', vector=[0.0, 0.85, 0.15, 0.0])]

## 3. Create a collection

The collection defines the vector size and distance metric. This cell recreates the collection so the notebook can be run repeatedly.

In [3]:
client = QdrantClient(":memory:") # http://localhost:6334 if you are quadrant in docker

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=models.VectorParams(
        size=VECTOR_SIZE,
        distance=models.Distance.COSINE,
    ),
)

True

## 4. Insert points

Store the vector separately from the searchable payload. The payload contains the title, text, and topic that we want returned with search results.

In [4]:
points = [
    models.PointStruct(
        id=resource.point_id,
        vector=resource.vector,
        payload={
            "title": resource.title,
            "text": resource.text,
            "topic": resource.topic,
        },
    )
    for resource in resources
]

client.upsert(collection_name=COLLECTION_NAME, points=points)
client.count(collection_name=COLLECTION_NAME, exact=True)

CountResult(count=4)

## 5. Search by vector similarity

The query vector represents a Python-related question. `query_points` returns the closest points according to cosine distance.

In [5]:
query_vector = [0.92, 0.08, 0.00, 0.00]
results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    limit=3,
    with_payload=True,
).points

for result in results:
    print(
        f"score={result.score:.3f} id={result.id} "
        f"title={result.payload['title']}"
    )

score=1.000 id=2 title=Python dictionaries
score=0.999 id=1 title=Python lists
score=0.139 id=3 title=Vector databases


## 6. Filter a vector search by payload

Vector similarity and structured filtering can be combined. This search only returns points whose `topic` payload is `python`.

In [6]:
python_only = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    query_filter=models.Filter(
        must=[
            models.FieldCondition(
                key="topic",
                match=models.MatchValue(value="python"),
            )
        ]
    ),
    limit=3,
    with_payload=True,
).points

for result in python_only:
    print(result.payload['title'], result.score)

Python dictionaries 0.9997139518004005
Python lists 0.9994167640063748


## 7. Update and retrieve a point

In [7]:
client.set_payload(
    collection_name=COLLECTION_NAME,
    payload={"level": "beginner"},
    points=[1],
)

record = client.retrieve(
    collection_name=COLLECTION_NAME,
    ids=[1],
    with_payload=True,
)[0]
record

Record(id=1, payload={'title': 'Python lists', 'text': 'Lists store an ordered collection of Python values.', 'topic': 'python', 'level': 'beginner'}, vector=None, shard_key=None, order_value=None)

## 8. Delete a point

Deleting by point ID removes the vector and its payload from the collection.

In [8]:
client.delete(
    collection_name=COLLECTION_NAME,
    points_selector=models.PointIdsList(points=[4]),
)

client.count(collection_name=COLLECTION_NAME, exact=True)

CountResult(count=3)